# Galaxy Multiplets Example

This notebook demonstrates how to use the `GalaxyMultiplets` class to identify galaxy multiplets (singlets, pairs, triplets, quadruplets) and compute their cross-correlations with the full galaxy sample.

## What are Galaxy Multiplets?

Multiplets are groups of galaxies that are spatially close to each other based on:
- **3D separation**: Distance in 3D space < `r_max` (default: 7 Mpc/h)
- **Projected separation**: Distance perpendicular to line-of-sight < `r_perp_max` (default: 1.05 Mpc/h)

Galaxies are linked together using a friends-of-friends style algorithm:
- Two galaxies that satisfy both criteria are linked
- All connected galaxies form a multiplet group
- Groups are classified by size: singlets (1), pairs (2), triplets (3), quadruplets (4), etc.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import fitsio
from acm.estimators.galaxy_clustering.multiplets import GalaxyMultiplets

## Load Galaxy Data

Load a galaxy catalog from HOD mocks. The catalog should contain positions in a periodic box.

In [ ]:
# Example: Load HOD catalog
filedir = f'/pscratch/sd/n/ntbfin/emulator/hods/z0.5/yuan23_prior/c000_ph000/seed0/'
hod, header = fitsio.read(f'{filedir}/hod000.fits', header=True)

# Get AP parameters
qpar, qperp = header['Q_PAR'], header['Q_PERP']

# Use z as the line-of-sight
pos = np.c_[hod['X_PERP'], hod['Y_PERP'], hod['Z_RSD']]
boxsize = np.array([2000/qperp, 2000/qperp, 2000/qpar])

# Ensure all galaxies are in [-L/2, L/2)
pos = np.mod(pos + boxsize/2, boxsize) - boxsize/2

# Shift to [0, L) for the estimator
coord = pos + boxsize[0]/2

print(f"Number of galaxies: {len(coord)}")
print(f"Boxsize: {boxsize}")
print(f"Position range: [{np.min(coord):.2f}, {np.max(coord):.2f}]")

## Initialize the GalaxyMultiplets Estimator

Create an instance with custom parameters for multiplet identification.

In [ ]:
# Initialize with default parameters
multiplets = GalaxyMultiplets(
    r_max=7.0,          # Maximum 3D separation (Mpc/h)
    r_perp_max=1.05,    # Maximum projected separation (Mpc/h)
    los='z',            # Line-of-sight direction
    nthreads=4          # Number of threads for correlation calculations
)

## Identify Multiplets

Run the complete pipeline to identify all multiplets in the catalog.

In [ ]:
# Identify all multiplets
results = multiplets.identify_multiplets(coord, boxsize)

print("\n=== Multiplet Statistics ===")
print(f"Singlets: {len(results['singlet_ids'])} ({len(results['singlet_ids'])/len(coord)*100:.2f}%)")
print(f"Pairs: {len(results['pair_coords'])}")
print(f"Triplets: {len(results['triplet_coords'])}")
print(f"Quadruplets: {len(results['quadruplet_coords'])}")
print(f"\nTotal multiplet groups: {len(results['groups_list'])}")
print(f"Galaxies in multiplets: {len(coord) - len(results['singlet_ids'])} ({(len(coord) - len(results['singlet_ids']))/len(coord)*100:.2f}%)")

## Multiplet Size Distribution

Visualize the distribution of multiplet sizes.

In [ ]:
# Plot multiplet size distribution
sizes = results['group_sizes']
unique_sizes, counts = np.unique(sizes, return_counts=True)

plt.figure(figsize=(5, 4))
plt.bar(unique_sizes, counts, alpha=0.7, edgecolor='black')
plt.xlabel('Multiplet Size', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Multiplet Sizes', fontsize=14)
plt.xticks(unique_sizes)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print statistics
print("Multiplet size distribution:")
for size, count in zip(unique_sizes, counts):
    print(f"  Size {size}: {count} groups ({count/len(sizes)*100:.2f}%)")

## Compute Cross-Correlations

Calculate the projected cross-correlation function between each multiplet type and the full galaxy sample.

In [ ]:
# Define binning for correlation function
edges = (
    np.linspace(2.0, 30.0, 14),   # rp bins
    np.linspace(-60, 60, 121)     # pi bins
)

# Compute all cross-correlations (projected correlation function wp)
correlations = multiplets.compute_all_cross_correlations(
    positions=coord,
    boxsize=boxsize,
    edges=edges,
    pimax=None  # Integrate over all pi (for wp calculation)
)

print("\nCross-correlation computed for:")
for key in correlations.keys():
    print(f"  - {key}")

## Visualize Cross-Correlation Functions

Plot the projected correlation function $w_p(r_p)$ for each multiplet type.

In [ ]:
# Calculate bin centers
rp_edges = edges[0]
rp_centers = (rp_edges[:-1] + rp_edges[1:]) / 2

# Plot wp(rp) for each multiplet type
plt.figure(figsize=(6, 4))

colors = {'singlet': 'blue', 'pair': 'orange', 'triplet': 'green', 'quadruplet': 'red'}
labels = {'singlet': 'Singlets', 'pair': 'Pairs', 'triplet': 'Triplets', 'quadruplet': 'Quadruplets'}

for key in ['singlet', 'pair', 'triplet', 'quadruplet']:
    if key in correlations:
        plt.plot(rp_centers, rp_centers * correlations[key], 
                marker='o', label=labels[key], color=colors[key], linewidth=2)

plt.xlabel(r'$r_p$ [Mpc/h]', fontsize=14)
plt.ylabel(r'$r_p \times w_p(r_p)$ [Mpc/h]$^2$', fontsize=14)
plt.title('Cross-Correlation: Multiplets × All Galaxies', fontsize=16)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.xscale('log')
plt.tight_layout()
plt.show()

## Create Summary Table

Generate a table with all correlation functions for easy saving and analysis.

In [ ]:
# Create summary table
summary = multiplets.get_summary_table(edges, correlations)

print("Summary table shape:", summary.shape)
print("Columns: [rp, wp_singlet, wp_pair, wp_triplet, wp_quadruplet]")
print("\nFirst few rows:")
print(summary[:5])

## Save Results

Save the correlation functions and multiplet information for later analysis.

In [ ]:
# Save summary table
# np.savetxt('multiplets_cross_correlations.txt', summary, 
#            header='rp wp_singlet wp_pair wp_triplet wp_quadruplet')

# Save multiplet information
# np.savez('multiplets_catalog.npz',
#          singlet_ids=results['singlet_ids'],
#          singlet_coords=results['singlet_coords'],
#          group_centers=results['group_centers'],
#          group_sizes=results['group_sizes'],
#          pair_coords=results['pair_coords'],
#          triplet_coords=results['triplet_coords'],
#          quadruplet_coords=results['quadruplet_coords'])

print("Results ready to save!")

## Advanced: Individual Correlation Function

Example of computing the correlation for a specific multiplet type manually.

In [ ]:
# Compute only for pairs
if len(results['pair_coords']) > 0:
    pair_estimator = multiplets.compute_cross_correlation(
        multiplet_coords=results['pair_coords'],
        all_positions=coord,
        boxsize=boxsize,
        edges=edges
    )
    
    # Get wp by integrating over pi
    wp_pairs = pair_estimator(pimax=None, return_sep=False)
    
    plt.figure(figsize=(6, 4))
    plt.plot(rp_centers, wp_pairs, 'o-', linewidth=2, markersize=8)
    plt.xlabel(r'$r_p$ [Mpc/h]', fontsize=14)
    plt.ylabel(r'$w_p(r_p)$ [Mpc/h]', fontsize=14)
    plt.title('Projected Correlation Function: Pairs × All Galaxies', fontsize=16)
    plt.grid(alpha=0.3)
    plt.xscale('log')
    plt.yscale('log')
    plt.tight_layout()
    plt.show()

## Spatial Distribution Visualization

Visualize the spatial distribution of different multiplet types (2D projection).

In [ ]:
# Create 2D projection (x-y plane)
fig, axes = plt.subplots(2, 2, figsize=(8, 8))

# Plot settings
plot_configs = [
    ('singlet_coords', 'Singlets', 'blue'),
    ('pair_coords', 'Pairs', 'orange'),
    ('triplet_coords', 'Triplets', 'green'),
    ('quadruplet_coords', 'Quadruplets', 'red')
]

for ax, (key, title, color) in zip(axes.flat, plot_configs):
    coords_key = results[key]
    if len(coords_key) > 0:
        # Plot a random subsample to avoid overcrowding
        n_plot = min(5000, len(coords_key))
        idx = np.random.choice(len(coords_key), n_plot, replace=False)
        ax.scatter(coords_key[idx, 0], coords_key[idx, 1], 
                  s=1, alpha=0.5, color=color)
    ax.set_xlabel('X [Mpc/h]', fontsize=12)
    ax.set_ylabel('Y [Mpc/h]', fontsize=12)
    ax.set_title(f'{title} (n={len(coords_key)})', fontsize=14)
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

The `GalaxyMultiplets` class provides a complete pipeline for:
1. Identifying galaxy multiplets based on spatial proximity
2. Computing multiplet centers accounting for periodic boundaries
3. Calculating cross-correlation functions between multiplets and all galaxies
4. Organizing results for analysis and visualization

### Key Methods:
- `identify_multiplets()`: Main pipeline for multiplet identification
- `compute_all_cross_correlations()`: Compute correlations for all multiplet types
- `get_summary_table()`: Format results as a table
- `compute_cross_correlation()`: Individual correlation for specific multiplet type

### Customization:
- Adjust `r_max` and `r_perp_max` for different linking criteria
- Change `los` parameter for different line-of-sight directions
- Modify binning in `edges` for desired resolution